In [1]:
import pandas as pd
from pandas.errors import PerformanceWarning
import warnings

warnings.simplefilter('ignore', category=PerformanceWarning)

In [2]:
from src.features.load_day_ahead_dataset import load_day_ahead_dataset

start = pd.Timestamp("2024-07-01")
end = pd.Timestamp("2025-04-01")
val_cutoff = pd.Timestamp("2025-01-01")
select_cutoff = pd.Timestamp("2025-02-01")
test_cutoff = pd.Timestamp("2025-03-01")

df = load_day_ahead_dataset(start, end)

2025-04-13 16:33:06,927 - INFO - Loading day ahead data
2025-04-13 16:33:06,927 - INFO - Loading data from /Users/pavelholman/FIT/bp/data/interim/data.csv


In [3]:
from src.models.data_split import time_series_split, DA_KNOWN
from src.models.train_day_ahead_time_series import train_day_ahead_time_series

ts_train, ts_val, ts_select, ts_test = time_series_split(df, val_cutoff, select_cutoff, test_cutoff)
ts_metrics = ["MAE", "RMSE"]

ts_mae_model = train_day_ahead_time_series(ts_train, ts_val, "MAE", cache="MAE")
display(ts_mae_model.leaderboard(ts_select, extra_metrics=ts_metrics))

ts_rmse_model = train_day_ahead_time_series(ts_train, ts_val, "RMSE", cache="RMSE")
display(ts_rmse_model.leaderboard(ts_select, extra_metrics=ts_metrics))

/Users/pavelholman/FIT/bp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-13 16:33:09,608 - INFO - Data split info:
Original DataFrame size: 6408
Train size: 4248 (66.29%)
Validation size: 744 (11.61%)
Select size: 672 (10.49%)
Test size: 744 (11.61%)
2025-04-13 16:33:09,608 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/time/MAE
Loading predictor from path /Users/pavelholman/FIT/bp/models/day_ahead/time/MAE
Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order,MAE,RMSE
0,Chronos[autogluon__chronos-bolt-small],-14.928996,-17.639528,0.716523,1.450066,1.903773,10,-14.928996,-19.310441
1,WeightedEnsemble,-15.417480,-16.595886,3.548968,6.220135,0.630978,11,-15.417480,-19.557700
2,AutoARIMA,-22.734620,-25.979802,1.566537,2.328330,0.148799,9,-22.734620,-27.514584
3,SeasonalNaive,-25.826071,-29.527202,0.036522,0.769916,0.005689,2,-25.826071,-31.671845
4,SeasonalAverage,-26.290357,-18.732411,0.075247,0.766376,0.003965,4,-26.290357,-33.397197
5,DirectTabular,-27.751879,-22.549799,2.718142,2.481918,46.311639,6,-27.751879,-33.226780
6,Average,-29.346905,-24.458095,0.054005,0.753477,0.004117,3,-29.346905,-35.819271
7,RecursiveTabular,-30.215719,-29.003966,7.092818,7.963616,49.514354,5,-30.215719,-36.471013
8,Naive,-45.320238,-42.927560,0.036881,1.521775,0.004528,1,-45.320238,-54.731337
9,ETS,-127.217593,-42.928406,0.229028,0.916793,0.036042,7,-127.217593,-152.315363


2025-04-13 16:33:11,710 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/time/RMSE
Loading predictor from path /Users/pavelholman/FIT/bp/models/day_ahead/time/RMSE
Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order,MAE,RMSE
0,Chronos[autogluon__chronos-bolt-small],-19.310441,-23.911603,0.688325,1.020571,0.003197,10,-14.928996,-19.310441
1,WeightedEnsemble,-20.164904,-22.830214,3.483781,3.991248,0.336087,11,-15.645940,-20.164904
2,AutoARIMA,-27.514584,-33.035764,1.102249,1.837500,0.051093,9,-22.734620,-27.514584
3,SeasonalNaive,-31.671845,-36.940488,0.033571,0.034290,0.003869,2,-25.826071,-31.671845
4,SeasonalAverage,-32.971903,-27.208264,0.076262,0.096332,0.005641,4,-25.471769,-32.971903
5,DirectTabular,-33.226780,-28.802748,2.667870,2.841976,48.937842,6,-27.751879,-33.226780
6,RecursiveTabular,-36.471013,-36.495478,7.053029,7.061484,47.679543,5,-30.215719,-36.471013
7,Average,-38.802135,-32.896616,0.051820,0.047948,0.004418,3,-32.155469,-38.802135
8,Naive,-54.731337,-52.172637,0.049631,0.032369,0.004853,1,-45.320238,-54.731337
9,ETS,-152.315363,-52.173367,0.208344,0.285673,0.027651,7,-127.217593,-152.315363


In [4]:
from src.models.data_split import tabular_split
from src.models.train_day_ahead_tabular import train_day_ahead_tabular

tab_df = df[[col for col in df.columns if any(col.startswith(prefix) for prefix in DA_KNOWN)] + ["DA Price (EUR/MWh)"]]
tab_train, tab_val, tab_select, tab_test = tabular_split(tab_df, val_cutoff, select_cutoff, test_cutoff)
tab_metrics = ["mae", "rmse"]

tab_mae_model = train_day_ahead_tabular(tab_train, tab_val, "mae", cache="MAE")
display(tab_mae_model.leaderboard(tab_test, extra_metrics=tab_metrics))

tab_rmse_model = train_day_ahead_tabular(tab_train, tab_val, "rmse", cache="RMSE")
display(tab_rmse_model.leaderboard(tab_test, extra_metrics=tab_metrics))

/Users/pavelholman/FIT/bp/src/models/data_split.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["item_id"] = 1
2025-04-13 16:33:13,933 - INFO - Data split info:
Original DataFrame size: 6408
Train size: 4248 (66.29%)
Validation size: 744 (11.61%)
Select size: 672 (10.49%)
Test size: 744 (11.61%)
2025-04-13 16:33:13,933 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/tabular/MAE


,model,score_test,mean_absolute_error,root_mean_squared_error,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGBoost,-21.071013,-21.071013,-28.805415,-26.141568,mean_absolute_error,0.283203,0.057007,44.012807,0.283203,0.057007,44.012807,1,True,4
1,WeightedEnsemble_L2,-21.346195,-21.346195,-28.302850,-22.087777,mean_absolute_error,0.726781,0.183402,414.389084,0.002124,0.000281,0.013841,2,True,5
2,CatBoost,-22.046666,-22.046666,-28.988917,-22.207982,mean_absolute_error,0.186502,0.049644,44.077024,0.186502,0.049644,44.077024,1,True,3
3,RandomForest,-25.159691,-25.159691,-34.371810,-25.797168,mean_absolute_error,0.254952,0.076470,326.285412,0.254952,0.076470,326.285412,1,True,2
4,KNeighbors,-31.617298,-31.617298,-42.312191,-38.519753,mean_absolute_error,0.224310,0.183607,1.467601,0.224310,0.183607,1.467601,1,True,1


2025-04-13 16:33:14,921 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/tabular/RMSE


,model,score_test,mean_absolute_error,root_mean_squared_error,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGBoost,-26.341249,-19.497605,-26.341249,-40.933329,root_mean_squared_error,0.070957,0.065134,66.932767,0.070957,0.065134,66.932767,1,True,4
1,CatBoost,-27.956053,-21.462751,-27.956053,-34.513444,root_mean_squared_error,0.120087,0.062885,94.734494,0.120087,0.062885,94.734494,1,True,3
2,WeightedEnsemble_L2,-27.956053,-21.462751,-27.956053,-34.513444,root_mean_squared_error,0.121447,0.063104,94.739043,0.001360,0.000219,0.004549,2,True,5
3,RandomForest,-34.371810,-25.159691,-34.371810,-40.374690,root_mean_squared_error,0.143845,0.067832,324.883975,0.143845,0.067832,324.883975,1,True,2
4,KNeighbors,-42.312191,-31.617298,-42.312191,-56.679368,root_mean_squared_error,0.337897,0.152940,0.421143,0.337897,0.152940,0.421143,1,True,1


In [5]:
from autogluon.timeseries import TimeSeriesDataFrame

ts_retrain = TimeSeriesDataFrame.from_data_frame(pd.concat([ts_train, ts_val, ts_select]))
ts_model = train_day_ahead_time_series(ts_retrain, None, "MAE", cache="retrain", models={"Chronos": {}})
display(ts_model.leaderboard(ts_test, extra_metrics=ts_metrics))

2025-04-13 16:33:15,837 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/time/retrain
Loading predictor from path /Users/pavelholman/FIT/bp/models/day_ahead/time/retrain
Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order,MAE,RMSE
0,Chronos[autogluon__chronos-bolt-small],-20.889781,-15.242769,0.663699,1.374355,2.932798,1,-20.889781,-28.203073
